# [INFO] Glu-Stock: 04_MONITOR_ALERT
**Phase**: Institutional Reporting & Telemetry (v18.2)

This notebook summarizes the day's events, calculates portfolio PnL, and reports system health to Telegram.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q pyTelegramBotAPI firebase-admin pandas yfinance psutil


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Helpers)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, psutil
from firebase_admin import credentials, firestore
from datetime import datetime
import telebot
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw_firebase = user_secrets.get_secret("FIREBASE_KEY_JSON")
                token = user_secrets.get_secret("TELEGRAM_TOKEN")
                chat_id = user_secrets.get_secret("TELEGRAM_CHAT_ID")
                return {"key": json.loads(raw_firebase), "token": token, "chat_id": chat_id}
            except Exception as e:
                print(f"[ERROR] Kaggle Secrets missing! Error: {e}")
                return {"key": None, "token": None, "chat_id": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "token": os.getenv("TELEGRAM_TOKEN"),
                "chat_id": os.getenv("TELEGRAM_CHAT_ID")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def get_latest_history(self, limit=10):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        return [doc.to_dict() for doc in docs]

    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]

    def wait_for_queue(self, queue_name: str, max_retries=10, interval=60):
        # Check if previous steps (03 Execution) finished. 
        # In 04, we don't clear queue, we just wait for the 'EXECUTION' event in history.
        import time
        print(f'[WAIT] Monitoring for completion of upstream tasks...', flush=True)
        for i in range(max_retries):
            latest = self.get_latest_history(limit=1)
            if latest and (latest[0]['phase'] in ['EXECUTION', 'CLOSER']):
                return True
            if i < max_retries - 1:
                print(f"[WAIT] Upstream not finished. Retrying ({i+1}/{max_retries}) in {interval}s...", flush=True)
                time.sleep(interval)
        return False


In [ ]:
# [MONITOR] SECTION 3: REPORTING LOGIC
def build_report(fb):
    # 1. Get History Summary
    history = fb.get_latest_history(5)
    hist_str = ""
    for h in history:
        hist_str += f"- [{h['phase']}] {h['details'][:40]}...\n"
    
    # 2. Get Active Portfolio
    active = fb.get_active_trades()
    port_str = ""
    total_unrealized_pnl = 0
    if active:
        for t in active:
            try:
                ticker = t['ticker']
                df = yf.download(ticker, period='1d', progress=False)
                curr = float(df['Close'].iloc[-1])
                pnl = (curr - t['entry_price']) * t['shares']
                total_unrealized_pnl += pnl
                status = "UP" if pnl >= 0 else "DWN"
                port_str += f"- {ticker}: {curr:,.0f} ({status} {pnl:,.0f})\n"
            except: pass
    else:
        port_str = "- No active positions.\n"
    
    # 3. Hardware Telemetry
    mem = psutil.virtual_memory()
    telemetry = f"RAM: {mem.percent}% | CPU: {psutil.cpu_percent()}%"

    # Final Message
    report = (
        f"**[GLU-STOCK] DAILY REPORT**\n"
        f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
        f"**LATEST ACTIVITY**:\n{hist_str}\n"
        f"**PORTFOLIO ({len(active)})**:\n{port_str}\n"
        f"**PNL UNREALIZED**: {total_unrealized_pnl:,.0f} IDR\n"
        f"**SYSTEM**: {telemetry}"
    )
    return report


In [ ]:
# [RUN] SECTION 4: MAIN EXECUTION
def run_monitor_cycle():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    # Wait for upstream sequence
    fb.wait_for_queue("signals") # Effectively waits for 03 to finish
    
    # Build & Send
    report_msg = build_report(fb)
    print("--- TELEGRAM REPORT ---")
    print(report_msg)
    
    if secrets.get('token') and secrets.get('chat_id'):
        try:
            bot = telebot.TeleBot(secrets['token'])
            bot.send_message(secrets['chat_id'], report_msg, parse_mode="Markdown")
            print("[OK] Report sent successfully.")
        except Exception as e: 
            print(f"[ERROR] Telegram upload failed: {e}")
    else:
        print("[WARN] Telegram credentials missing in Kaggle Secrets.")
        
run_monitor_cycle()